# 01 — Data Preparation

**Goal:** Load Banking77 from HuggingFace, audit data quality, carve fixed validation + dev splits from the train pool, and sample the stratified training subsets used by the fine-tune sweep in notebook 03.

## Sections

1. Load Banking77 via `datasets.load_dataset`
2. Schema audit + class-balance check
3. Query-length distribution (informs `max_length` for DistilBERT)
4. Confirm test holdout (Banking77 ships with an official test split — use it)
5. Carve fixed validation set (500 rows, stratified) from train pool
6. Carve dev slice (300 rows, stratified) from train pool — used for LLM prompt iteration in notebook 02
7. Sample stratified training subsets at n ∈ {50, 100, 250, 500, 1000, 2500, 5000} × 5 random seeds
8. Save processed splits to `data/processed/`

Why val + dev come from the train pool, not the test set: keeps the final test eval uncontaminated. The dev slice is used to iterate the LLM prompt; if it shared rows with test, the final test metrics would be optimistically biased.

## 1. Load Banking77

In [ ]:
import pandas as pd
from datasets import load_dataset
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ds = load_dataset('PolyAI/banking77')
print(ds)

# Convert to pandas for easier inspection
train_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()
label_names = ds['train'].features['label'].names
print(f'Number of classes: {len(label_names)}')
print(f'Train size: {len(train_df)}; test size: {len(test_df)}')

## 2. Schema audit + class-balance check

In [ ]:
# TODO: nulls, dtypes, class-count histogram
# Watch for intents with < 50 train examples — these will bite at small n

## 3. Query-length distribution

Banking77 queries are typically short (banking chatbot interactions). Verify the actual distribution so we can choose `MAX_LENGTH` for DistilBERT — likely 64 or 128 is enough, not the default 512.

In [ ]:
# TODO: token-length histogram (use the DistilBERT tokenizer for accurate count)

## 4. Test holdout

Banking77 has an official test split (3,080 rows). Use it — comparable to published baselines. This set is **only** touched by the final eval in notebook 02 and the DistilBERT evals in notebook 03. Never iterate on it.

In [ ]:
# TODO: confirm test-set class distribution roughly matches train

## 5. Carve fixed validation set

Carve **500 stratified rows** from the 10,003-row train pool as a fixed val set. The same 500 rows are used as val for every n and every seed in notebook 03 — this keeps early-stopping decisions comparable across the sweep.

After this carve-out the remaining train pool is ~9,503 rows, from which n-subsets are sampled in section 7.

In [ ]:
from sklearn.model_selection import train_test_split

VAL_SET_SIZE = 500
RANDOM_SEED = 42

# TODO: stratified split — train_pool, val = train_test_split(train_df, test_size=VAL_SET_SIZE, stratify=train_df['label'], random_state=RANDOM_SEED)
# TODO: save val to data/processed/val.parquet

## 6. Carve dev slice for LLM prompt iteration

Carve **300 stratified rows** from the remaining train pool as the dev slice. Used in notebook 02 for iterating the LLM prompt at low cost (~$1.50 per Sonnet 4.6 eval at this size).

After this carve-out the remaining train pool is ~9,203 rows, more than enough for n up to 5,000.

In [ ]:
DEV_SLICE_SIZE = 300

# TODO: stratified split off the remaining train_pool — train_pool, dev = train_test_split(train_pool, test_size=DEV_SLICE_SIZE, stratify=train_pool['label'], random_state=RANDOM_SEED)
# TODO: save dev to data/processed/dev_slice.parquet

## 7. Sample stratified training subsets across seeds

For each (n, seed) pair, draw a stratified random subset of size n from the remaining train pool. **Subsets are nested within seed** — for seed=0, the n=100 subset is a superset of the n=50 subset etc. — so comparisons across n at the same seed isolate the effect of adding training data.

Note on stratification at n=50: with 77 classes and 50 examples, perfect stratification (one per class) is impossible. The sampler will fall back to whatever distribution is achievable; track which intents get zero training examples per (n, seed) and report this in the analysis notebook.

In [ ]:
TRAINING_SIZES = [50, 100, 250, 500, 1000, 2500, 5000]
N_SEEDS = 5

# TODO: for each seed in 0..N_SEEDS:
#         shuffle train_pool with this seed
#         for each n in TRAINING_SIZES (ascending):
#             take the first n rows (stratified where possible) — nested by construction
#             save to data/processed/train_n{n}_seed{seed}.parquet

## 8. Save processed splits

In [ ]:
# TODO: save test_df to data/processed/test.parquet
# TODO: save label_names to data/processed/label_names.txt (one per line, index = label id)

## Notes

- *(observations, decisions, surprises — fill in as you go)*
- Train pool budget: 10,003 → minus 500 val → minus 300 dev → 9,203 available for n-subsets. n=5,000 still fits comfortably.
- Stratification at small n with 77 classes can't be perfect (n=50 < 77). Log which intents are missing per (n, seed) and report in notebook 04.